# Lab 1 — The Judgement Machine: Personal Statement Roast

### AI for Teens | Agentic AI with CrewAI

---

You are about to build a team of three AI agents that read a university personal statement and tell you the truth about it.

**Your crew:**

| Agent | Their job |
|---|---|
| **The Admissions Officer** | Has read 4,000 statements this year. Bored. Blunt. |
| **The Cliche Detector** | Hunts down every "from a young age I have always been passionate about..." |
| **The Fixer** | Takes your worst paragraph and rewrites it properly |

**The rule they all follow:** they attack the *writing*, never the *writer*.

By the end of this lab your crew will be running as a real web app that anyone with the link can use.

**Time:** ~75 minutes

---

### Before you start

Go to **Runtime -> Change runtime type** and make sure it says **CPU**. You do not need a GPU. Using one just wastes your free Colab time.

## Step 1 — Install CrewAI

This takes 3-5 minutes. CrewAI brings a lot of friends along with it.

Run the cell, then **read Step 2 while it installs**. Do not sit and watch the bar.

In [ ]:
%pip install crewai crewai-tools -q

print("Install finished.")

### Restart the runtime

Colab already had some of these libraries loaded before we upgraded them, so it is still using the old versions in memory.

**Run the cell below.** It will crash the session on purpose. That is correct. A popup will say the session crashed — click **OK** and carry on to Step 2.

> Do NOT re-run Step 1 after the restart. The install is saved.

In [ ]:
import os
os.kill(os.getpid(), 9)

## Step 2 — Your API key

Your agents need a brain. We are using OpenAI's `gpt-4o-mini` — fast and cheap.

### Putting the key in safely

Never paste an API key straight into a code cell. If you share the notebook, you share your key, and whoever has it can spend your money.

Colab has a proper place for secrets:

1. Click the **key icon** in the far-left sidebar
2. Click **+ Add new secret**
3. Name: `OPENAI_API_KEY`
4. Value: paste your key
5. Turn **Notebook access** ON (the toggle on the left)

Then run the cell below.

In [ ]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["OPENAI_MODEL_NAME"] = "gpt-4o-mini"

key = os.environ["OPENAI_API_KEY"]
assert key.startswith("sk-"), "That does not look like an OpenAI key."
print(f"Key loaded: {key[:7]}...{key[-4:]}")
print("Model:", os.environ["OPENAI_MODEL_NAME"])

## Step 3 — One agent, one task

Before we build a crew of three, let us prove that a single agent works.

Every agent in CrewAI is built from exactly three things:

- **role** — what they are
- **goal** — what they are trying to achieve
- **backstory** — who they are, which shapes *how* they do it

That is it. The backstory is not decoration. It is the strongest lever you have over how the agent behaves.

In [ ]:
from crewai import Agent, Task, Crew

test_agent = Agent(
    role="Personal Statement Reader",
    goal="Say in one sentence what this statement is really about",
    backstory=(
        "You cut through waffle. You have no patience for long "
        "introductions and you always answer in a single sentence."
    ),
    verbose=True
)

test_task = Task(
    description=(
        "Read this opening line from a personal statement and say what it is "
        "actually about, in ONE sentence:\n\n"
        "'From a young age I have always been passionate about the "
        "intricate workings of the human body and the noble pursuit of medicine.'"
    ),
    expected_output="One single sentence. No preamble.",
    agent=test_agent
)

test_crew = Crew(agents=[test_agent], tasks=[test_task], verbose=True)
print(test_crew.kickoff())

### What just happened

Scroll up through that output. You can see the agent *thinking* before it answers — that is the `verbose=True` setting showing you its working.

**Try this:** change the backstory to `"You are extremely polite and always find something nice to say."` and run the cell again.

Same task. Same model. Completely different answer.

That is the whole idea of this lab. You are not writing instructions for a program — you are hiring a personality.

## Step 4 — Build the crew

Now the real three. Read each backstory carefully before you run this cell.

Notice what every one of them contains: a line telling the agent to criticise the **writing**, not the **writer**. We are not hoping the model is decent about this. We are instructing it.

In [ ]:
from crewai import Agent

FAIRNESS_RULE = (
    "CRITICAL RULE: You criticise the WRITING, never the PERSON. "
    "Say 'this sentence says nothing specific', never 'you are boring'. "
    "Say 'this claim has no evidence behind it', never 'you are lazy'. "
    "The writer is a teenager applying to university. Be sharp about the "
    "text and never cruel about them."
)

admissions_officer = Agent(
    role="University Admissions Officer",
    goal="Judge whether this statement would survive a real admissions pile",
    backstory=(
        "You have read 4,000 personal statements this year alone. You read each "
        "one for about 45 seconds. You have seen every opening line a thousand "
        "times and you are extremely hard to impress. You care about one thing: "
        "does this applicant show me evidence, or are they just telling me they "
        "are passionate? You are blunt because sugar-coating helps nobody. "
        + FAIRNESS_RULE
    ),
    verbose=True
)

cliche_detector = Agent(
    role="Cliche Detector",
    goal="Find every tired, overused, or empty phrase in the statement",
    backstory=(
        "You are a machine built for one purpose: spotting phrases that appear "
        "in thousands of other statements. 'From a young age.' 'I have always "
        "been passionate about.' 'This sparked my interest.' 'In today's "
        "ever-changing world.' You also catch empty claims — words like "
        "'hardworking' or 'dedicated' with no proof attached. You quote the "
        "exact phrase, then explain what is wrong with it. "
        + FAIRNESS_RULE
    ),
    verbose=True
)

fixer = Agent(
    role="Writing Coach",
    goal="Rewrite the weakest paragraph so it actually works",
    backstory=(
        "You are the one who helps after the damage is done. You take the "
        "single weakest paragraph and rebuild it — swapping vague claims for "
        "specific evidence, cutting filler, keeping the writer's own voice. "
        "You never invent facts about the student. If a claim needs evidence "
        "you do not have, you write [ADD SPECIFIC EXAMPLE HERE] so they can "
        "fill it in themselves. "
        + FAIRNESS_RULE
    ),
    verbose=True
)

print("Three agents hired.")

## Step 5 — Give them tasks

An agent is a *who*. A task is a *what*.

The `expected_output` field matters more than students usually expect — it is how you control the shape of what comes back. Vague expected output, vague result.

Notice `context=[...]` on the last task. That is how the Fixer gets to see what the other two found. Without it, all three would work in isolation and repeat each other.

In [ ]:
from crewai import Task

screen_task = Task(
    description=(
        "Read this personal statement for the course: {course}\n\n"
        "STATEMENT:\n{statement}\n\n"
        "Judge it as you would in a real admissions pile. Give a score out of "
        "10 and justify it. Name the single biggest problem with it."
    ),
    expected_output=(
        "Format exactly like this:\n"
        "SCORE: X/10\n"
        "VERDICT: two sentences on whether this survives the pile\n"
        "BIGGEST PROBLEM: one paragraph"
    ),
    agent=admissions_officer
)

cliche_task = Task(
    description=(
        "Hunt through this statement for cliches and empty claims.\n\n"
        "STATEMENT:\n{statement}\n\n"
        "Find the worst offenders. For each one, quote the exact phrase and "
        "explain in one line why it is weak."
    ),
    expected_output=(
        "A numbered list of 3 to 5 items. Each item:\n"
        '1. "exact quoted phrase" -> why it is weak (one line)'
    ),
    agent=cliche_detector
)

fix_task = Task(
    description=(
        "Using what the other two found, pick the SINGLE weakest paragraph "
        "from this statement and rewrite it.\n\n"
        "STATEMENT:\n{statement}\n\n"
        "Course applied for: {course}\n\n"
        "Show the original, then your rewrite, then explain what you changed."
    ),
    expected_output=(
        "ORIGINAL:\n[the paragraph as written]\n\n"
        "REWRITTEN:\n[your version]\n\n"
        "WHAT CHANGED:\n[3 bullet points]"
    ),
    agent=fixer,
    context=[screen_task, cliche_task]
)

print("Three tasks assigned.")

## Step 6 — Assemble and run

`Process.sequential` means they work in order: Officer, then Detector, then Fixer.

The statement below is deliberately bad. It is stuffed with every cliche in the book. Run it first so you can see the crew tear it apart, then swap in a real one.

In [ ]:
from crewai import Crew, Process

roast_crew = Crew(
    agents=[admissions_officer, cliche_detector, fixer],
    tasks=[screen_task, cliche_task, fix_task],
    process=Process.sequential,
    verbose=True
)

BAD_STATEMENT = """
From a young age I have always been passionate about computers and technology.
In today's ever-changing world, technology is more important than ever before.
I am a hardworking and dedicated student who always gives 100% to everything I do.

My interest in Computer Science was sparked when I got my first laptop. Since then
I have been fascinated by how things work behind the screen. I enjoy problem
solving and I believe I have strong analytical skills.

At school I study Maths, Physics and Computer Science. I am also a member of the
school football team, which has taught me valuable teamwork skills. I believe
these skills will help me greatly at university.

In conclusion, I am confident that I would be an excellent addition to your
university and I look forward to the opportunity.
"""

result = roast_crew.kickoff(inputs={
    "statement": BAD_STATEMENT,
    "course": "Computer Science"
})

print("\n" + "="*60)
print(result)

### Now try a real one

Replace `BAD_STATEMENT` above with an actual personal statement — yours, a sibling's, or one you find online — change the course, and run it again.

**Watch for:** does the crew say anything genuinely useful, or does it just sound harsh? If the feedback is generic, the problem is almost always the backstory, not the model. Go back to Step 4 and make the agent more specific about what it cares about.

---

# Part 2 — Make it a real app

Right now your crew only works if someone opens this notebook. Let us give it a web interface and a link you can send to anyone.

**Three moving parts:**

1. **Streamlit** — turns Python into a web page
2. **A tunnel** — Colab is a locked box with no public door; the tunnel makes one
3. **Your crew** — moved into a file called `app.py`

In [ ]:
%pip install streamlit -q

# Download cloudflared - this is what gives us a public link
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

print("Streamlit and cloudflared ready.")

## Step 7 — Write the app

`%%writefile` at the top of a cell means "do not run this code — save it to a file instead."

Everything below gets written to `app.py`. That is why your agents are defined again here: `app.py` is a separate program and knows nothing about this notebook.

Read through it. It is the same crew you already built, wrapped in about 30 lines of interface.

In [ ]:
%%writefile app.py
import os
import streamlit as st
from crewai import Agent, Task, Crew, Process

st.set_page_config(page_title="The Judgement Machine", page_icon="X", layout="wide")

FAIRNESS_RULE = (
    "CRITICAL RULE: You criticise the WRITING, never the PERSON. "
    "Say 'this sentence says nothing specific', never 'you are boring'. "
    "The writer is a teenager applying to university. Be sharp about the "
    "text and never cruel about them."
)


def build_crew():
    officer = Agent(
        role="University Admissions Officer",
        goal="Judge whether this statement would survive a real admissions pile",
        backstory=(
            "You have read 4,000 personal statements this year. You read each for "
            "45 seconds. You are extremely hard to impress and you care about one "
            "thing: evidence, not claims of passion. " + FAIRNESS_RULE
        ),
    )
    detector = Agent(
        role="Cliche Detector",
        goal="Find every tired, overused, or empty phrase",
        backstory=(
            "You spot phrases that appear in thousands of other statements, and "
            "empty claims like 'hardworking' with no proof attached. You quote the "
            "exact phrase, then explain what is wrong with it. " + FAIRNESS_RULE
        ),
    )
    coach = Agent(
        role="Writing Coach",
        goal="Rewrite the weakest paragraph so it actually works",
        backstory=(
            "You rebuild the weakest paragraph - vague claims become specific "
            "evidence, filler is cut, the writer's voice stays. You never invent "
            "facts; you write [ADD SPECIFIC EXAMPLE HERE] instead. " + FAIRNESS_RULE
        ),
    )

    t1 = Task(
        description=(
            "Read this statement for the course: {course}\n\n{statement}\n\n"
            "Judge it as in a real admissions pile. Score out of 10, justify it, "
            "name the single biggest problem."
        ),
        expected_output="SCORE: X/10\nVERDICT: two sentences\nBIGGEST PROBLEM: one paragraph",
        agent=officer,
    )
    t2 = Task(
        description=(
            "Hunt for cliches and empty claims in this statement:\n\n{statement}\n\n"
            "Quote each exact phrase and explain in one line why it is weak."
        ),
        expected_output='Numbered list of 3-5 items: "phrase" -> why it is weak',
        agent=detector,
    )
    t3 = Task(
        description=(
            "Pick the SINGLE weakest paragraph and rewrite it.\n\n{statement}\n\n"
            "Course: {course}"
        ),
        expected_output="ORIGINAL:\n...\n\nREWRITTEN:\n...\n\nWHAT CHANGED:\n3 bullets",
        agent=coach,
        context=[t1, t2],
    )

    return Crew(agents=[officer, detector, coach], tasks=[t1, t2, t3],
                process=Process.sequential, verbose=True)


st.title("The Judgement Machine")
st.caption("Three AI agents read your personal statement and tell you the truth.")
st.info("They criticise the writing, never the writer.")

col1, col2 = st.columns([2, 1])

with col2:
    course = st.text_input("Course applying for", value="Computer Science")
    st.markdown("**Your crew:**")
    st.markdown("- Admissions Officer - scores it\n- Cliche Detector - finds the waffle\n- Writing Coach - fixes the worst bit")

with col1:
    statement = st.text_area("Paste your personal statement", height=320,
                             placeholder="Paste the full statement here...")

if st.button("Judge it", type="primary"):
    if len(statement.strip()) < 100:
        st.warning("That is too short - paste the full statement.")
    else:
        with st.spinner("Your crew is reading. This takes 30-60 seconds..."):
            try:
                crew = build_crew()
                result = crew.kickoff(inputs={"statement": statement, "course": course})
                st.success("Verdict in.")
                st.markdown("---")
                st.markdown(str(result))
            except Exception as e:
                st.error(f"Something broke: {e}")
                st.caption("Most common cause: the API key did not carry over. Re-run the launch cell.")


## Step 8 — Launch it

This cell starts the server and opens the tunnel.

**What to look for:** a few seconds after running, a link ending in **`.trycloudflare.com`** appears in the output. That is your app. Click it.

> First load takes 10-20 seconds. If you get an error page, wait and refresh once.

In [ ]:
import os, time, subprocess, threading, re

# Pass the API key through to app.py
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["OPENAI_MODEL_NAME"] = "gpt-4o-mini"

# Kill anything left over from a previous run
!pkill -f streamlit >/dev/null 2>&1
!pkill -f cloudflared >/dev/null 2>&1
time.sleep(2)

# Start Streamlit in the background
subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, env=os.environ.copy()
)
time.sleep(8)

# Open the tunnel and watch for the link
proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

print("Opening tunnel...\n")
for line in proc.stdout:
    found = re.search(r"https://[-\w]+\.trycloudflare\.com", line)
    if found:
        print("=" * 60)
        print("YOUR APP IS LIVE:")
        print(found.group(0))
        print("=" * 60)
        print("\nKeep this cell running. Stopping it kills the app.")
        break

### It is live

That link works for anyone, anywhere, right now. Send it to someone.

**Two things to know:**

- The link dies when you stop the cell or the Colab session ends. That is normal for a free tunnel.
- Every run costs a small amount of your OpenAI credit. Roughly a fraction of a cent per statement with `gpt-4o-mini`, but do not leave it open to the whole internet overnight.

---

## When it goes wrong

| What you see | Why | Fix |
|---|---|---|
| `ModuleNotFoundError: crewai` | You skipped the restart, or re-ran Step 1 after it | Restart runtime, then run Step 2 onwards |
| `SecretNotFoundError` | Notebook access toggle is off | Key icon in sidebar, turn the toggle on |
| `AuthenticationError` | Wrong key, or extra spaces when pasting | Re-copy the key, no spaces |
| `RateLimitError` / `insufficient_quota` | No credit on the OpenAI account | Add credit at platform.openai.com |
| No `.trycloudflare.com` link appears | Streamlit failed to start | Run `!streamlit run app.py` on its own to see the real error |
| App loads but button does nothing | Key did not reach `app.py` | Stop the cell, re-run Step 2, re-run the launch cell |
| Feedback is generic and boring | Backstories are too vague | Go back to Step 4 and be far more specific |

---

## Challenges

Finished early? Pick one.

**1. Add a fourth agent.** A **Course Fit Checker** who judges whether the statement matches the course being applied for. Where does it go in the sequence, and what does it need in its `context`?

**2. Make the score adjustable.** Add a Streamlit slider for "brutality: 1-10" and feed it into the Officer's backstory. What actually changes in the output?

**3. Break it on purpose.** Delete the `FAIRNESS_RULE` from all three agents and run it on the bad statement again. Read the difference carefully. Then put it back. That difference is the single most important thing in this lab.

**4. Change the domain entirely.** Same three-agent shape, completely different subject: roast a CV, a project proposal, a song lyric. How much of the code has to change? (Answer: less than you think.)